# Inference from Saved `best_checkpoint` (Load from Google Drive)

This notebook runs **inference only** using a previously saved Whisper `best_checkpoint` from Google Drive.

## Step-by-step setup (important)
1. Start a fresh Google Colab runtime.
2. Ensure your trained folder exists in Drive, for example:
   - `/content/drive/MyDrive/AAI3008_artifacts_exports/best_checkpoint`
3. Upload your input audio file to `/content` (for example `/content/input_audio.m4a`) or put audio in Drive and set the path accordingly.
4. In the config cell, set:
   - `DRIVE_CHECKPOINT_DIR`
   - `AUDIO_PATH`
5. Run all cells from top to bottom.

## What this notebook prints
1. Audio metrics (duration, chunk count, detected language distribution)
2. Full transcription text
3. Optional per-chunk details table


In [ ]:
# Colab dependency bootstrap (fixes numpy/scipy binary mismatch)
# Run this cell once after connecting runtime. It will restart runtime automatically.
%pip install -q --upgrade --force-reinstall "numpy==2.2.2" "scipy==1.15.1"
%pip install -q --upgrade "transformers>=4.46" "torch" "pydub>=0.25" "pandas>=2.0" "accelerate>=0.34"

import os
os.kill(os.getpid(), 9)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 14.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.4 MB/s eta 0:00:00
ERR

In [ ]:
# Optional helper: upload files directly from your machine to /content
# Uncomment and run if needed.
# from google.colab import files
# files.upload()


In [ ]:
from __future__ import annotations

import re
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from pydub import AudioSegment
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f'Drive mount skipped/unavailable: {e}')

# ===== User-configurable variables =====
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/University/Y2T2/LLM/AAI3008_artifacts/best_checkpoint_v4'
#AUDIO_PATH = ['/content/000010001.WAV','/content/000010002.WAV', '/content/000010003.WAV', '/content/000010005.WAV', '/content/000010006.WAV' ]  # can also be a Drive path like /content/drive/MyDrive/.../audio.m4a
AUDIO_PATH = '/content/tsl_audio_test.m4a'
REFERENCE_TEXT_PATH = None            # optional: '/content/reference_transcript.txt'

# Model choices (baseline comparison)
BASELINE_MODEL_ID = 'openai/whisper-small'  # try 'openai/whisper-medium' or 'openai/whisper-large-v3'

# Inference settings
CHUNK_SEC = 20.0
CHUNK_OVERLAP_SEC = 2.0
MIN_CHUNK_SEC = 1.5
MAX_DECODE_LEN = 384
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Inference mode: 'auto' | 'en' | 'zh' | 'mixed'
INFERENCE_MODE = 'auto'

# Optional per-chunk language hints: {chunk_idx: 'en'|'zh'|'mixed'}
CHUNK_LANG_HINTS = None

# Overlap merge settings (reduces repeated text at chunk boundaries)
MERGE_OVERLAP = True
OVERLAP_MAX_TOKENS = 20
OVERLAP_MIN_TOKENS = 4

# Timestamped outputs (for PII bleep alignment)
RETURN_TIMESTAMPS = True
SAVE_SEGMENT_FILES = True

print(f'DEVICE={DEVICE}')
print(f'DRIVE_CHECKPOINT_DIR={DRIVE_CHECKPOINT_DIR}')
print(f'AUDIO_PATH={AUDIO_PATH}')
print(f'REFERENCE_TEXT_PATH={REFERENCE_TEXT_PATH}')
print(f'BASELINE_MODEL_ID={BASELINE_MODEL_ID}')

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Mounted at /content/drive
DEVICE=cuda
DRIVE_CHECKPOINT_DIR=/content/drive/MyDrive/University/Y2T2/LLM/AAI3008_artifacts/best_checkpoint_v4
AUDIO_PATH=/content/tsl_audio_test.m4a
REFERENCE_TEXT_PATH=None


In [ ]:
def resolve_checkpoint_dir(source: str) -> Path:
    src = Path(source)
    if not src.exists():
        raise FileNotFoundError(f'Checkpoint source not found: {src}')

    if src.is_dir():
        return src

    if src.suffix.lower() == '.zip':
        target = Path('/content/best_checkpoint')
        if target.exists():
            # reuse existing extracted folder
            return target
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src, 'r') as zf:
            zf.extractall(target)

        # Sometimes zip contains a nested best_checkpoint folder.
        nested = target / 'best_checkpoint'
        if nested.exists() and nested.is_dir():
            return nested
        return target

    raise ValueError('CHECKPOINT_SOURCE must be a directory or .zip file')


def load_audio_mono_16k(path: str):
    audio = AudioSegment.from_file(path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    arr = np.array(audio.get_array_of_samples())
    max_val = float(1 << (8 * audio.sample_width - 1))
    arr = arr.astype(np.float32) / max_val
    return arr, 16000


# --- Overlap-aware chunking ---
def chunk_ranges(total_sec: float, chunk_sec: float, min_chunk_sec: float):
    spans = []
    step = max(0.5, chunk_sec - CHUNK_OVERLAP_SEC)
    s = 0.0
    while s < total_sec:
        e = min(s + chunk_sec, total_sec)
        if e - s >= min_chunk_sec:
            spans.append((float(s), float(e)))
        if e >= total_sec:
            break
        s += step
    return spans


_ZH_RE = re.compile(r'[\u4e00-\u9fff]')
_LAT_RE = re.compile(r'[A-Za-z]')


def has_zh(t: str) -> bool:
    return bool(_ZH_RE.search(t or ''))


def has_lat(t: str) -> bool:
    return bool(_LAT_RE.search(t or ''))


def language_bucket(text: str) -> str:
    zh = has_zh(text)
    lat = has_lat(text)
    if zh and lat:
        return 'mixed'
    if zh:
        return 'zh'
    if lat:
        return 'en'
    return 'unknown'


MALAY_HINT_WORDS = {
    'saya','dan','yang','tidak','untuk','ini','itu','dengan','anda','terima','kasih','boleh','akan'
}


def _malay_hint_score(text: str) -> int:
    toks = re.findall(r"[A-Za-z']+", (text or '').lower())
    return sum(1 for t in toks if t in MALAY_HINT_WORDS)


def choose_mixed(en_text: str, zh_text: str) -> tuple[str, str]:
    # Prefer candidates preserving Chinese while still allowing Latin.
    # Penalize Malay/Indonesian-like lexical hints in Latin output.
    def score(t: str) -> int:
        s = 0
        if has_zh(t):
            s += 4
        if has_lat(t):
            s += 2
        s -= 2 * _malay_hint_score(t)
        s += min(2, len((t or '').strip()) // 20)
        return s

    s_en = score(en_text)
    s_zh = score(zh_text)
    if s_zh > s_en:
        return zh_text, 'zh_prompt'
    if s_en > s_zh:
        return en_text, 'en_prompt'

    # Tie-breaker: preserve Chinese if present in one candidate only.
    if has_zh(zh_text) and not has_zh(en_text):
        return zh_text, 'zh_prompt'
    if has_zh(en_text) and not has_zh(zh_text):
        return en_text, 'en_prompt'

    return zh_text, 'zh_prompt'


def resolve_chunk_lang(idx: int, default_mode: str) -> str:
    mode = (default_mode or 'auto').lower()
    if isinstance(CHUNK_LANG_HINTS, dict):
        hint = CHUNK_LANG_HINTS.get(idx)
        if hint:
            mode = str(hint).lower()
    return mode


def validate_checkpoint_dir(path_str: str) -> Path:
    # Compatibility wrapper: directory-based validation/load
    return resolve_checkpoint_dir(path_str)


# --- Overlap merge helpers ---
def _tokenize_for_overlap(text: str) -> list[str]:
    t = (text or '').strip()
    if not t:
        return []
    if has_zh(t):
        return [c for c in t if not c.isspace()]
    return re.findall(r"\S+", t)


def _overlap_prefix(prev_tokens: list[str], curr_tokens: list[str], max_tokens: int, min_tokens: int) -> int | None:
    max_k = min(max_tokens, len(prev_tokens), len(curr_tokens))
    for k in range(max_k, min_tokens - 1, -1):
        if prev_tokens[-k:] == curr_tokens[:k]:
            return k
    return None


def _remove_prefix_text(curr_text: str, overlap_str: str) -> str:
    if not overlap_str:
        return curr_text
    if curr_text.startswith(overlap_str):
        return curr_text[len(overlap_str):].lstrip()
    stripped = curr_text.lstrip()
    if stripped.startswith(overlap_str):
        return stripped[len(overlap_str):].lstrip()
    return curr_text


def merge_overlapping_texts(chunks: list[str], max_tokens: int = 20, min_tokens: int = 4) -> str:
    merged = []
    for text in chunks:
        if not text:
            continue
        if not merged:
            merged.append(text.strip())
            continue
        prev_text = merged[-1]
        prev_tokens = _tokenize_for_overlap(prev_text)
        curr_tokens = _tokenize_for_overlap(text)
        k = _overlap_prefix(prev_tokens, curr_tokens, max_tokens, min_tokens)
        if k:
            if has_zh(text) or has_zh(prev_text):
                overlap_str = ''.join(curr_tokens[:k])
            else:
                overlap_str = ' '.join(curr_tokens[:k])
            text = _remove_prefix_text(text, overlap_str)
        merged.append(text.strip())
    return ' '.join(t for t in merged if t).strip()


# --- Timestamp helpers ---
_TS_RE = re.compile(r'^<\|([0-9]+(?:\.[0-9]+)?)\|>$')


def decode_chunk_with_timestamps(proc, mdl, chunk_audio: np.ndarray, sr: int, prompt_lang: str,
                                 chunk_start: float, chunk_end: float, max_decode_len: int | None = None):
    if max_decode_len is None:
        max_decode_len = MAX_DECODE_LEN

    feat = proc.feature_extractor(
        chunk_audio,
        sampling_rate=sr,
        return_tensors='pt',
        return_attention_mask=True,
    )
    input_features = feat.input_features.to(mdl.device)
    attention_mask = feat.attention_mask.to(mdl.device)

    if prompt_lang == 'en':
        forced_decoder_ids = proc.tokenizer.get_decoder_prompt_ids(language='english', task='transcribe')
    elif prompt_lang == 'zh':
        forced_decoder_ids = proc.tokenizer.get_decoder_prompt_ids(language='chinese', task='transcribe')
    else:
        forced_decoder_ids = proc.tokenizer.get_decoder_prompt_ids(task='transcribe')

    with torch.no_grad():
        pred_ids = mdl.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            forced_decoder_ids=forced_decoder_ids,
            max_length=max_decode_len,
            num_beams=5,
            do_sample=False,
            return_timestamps=True,
        )

    tokens = proc.tokenizer.convert_ids_to_tokens(pred_ids[0])
    segments = []
    cur_start = None
    cur_tokens: list[str] = []

    for tok in tokens:
        m = _TS_RE.match(tok)
        if m:
            ts = float(m.group(1))
            if cur_start is None:
                cur_start = ts
            else:
                text = proc.tokenizer.convert_tokens_to_string(cur_tokens).strip()
                if text:
                    segments.append({
                        'start_sec': chunk_start + cur_start,
                        'end_sec': chunk_start + ts,
                        'text': text,
                    })
                cur_start = ts
                cur_tokens = []
            continue

        # Skip other special tokens
        if tok.startswith('<|') and tok.endswith('|>'):
            continue
        cur_tokens.append(tok)

    if cur_tokens and cur_start is not None:
        text = proc.tokenizer.convert_tokens_to_string(cur_tokens).strip()
        if text:
            end_ts = min(chunk_end - chunk_start, cur_start + 30.0)
            segments.append({
                'start_sec': chunk_start + cur_start,
                'end_sec': chunk_start + end_ts,
                'text': text,
            })

    return segments


def _format_srt_ts(sec: float) -> str:
    ms = int(round(sec * 1000))
    h = ms // 3600000
    ms -= h * 3600000
    m = ms // 60000
    ms -= m * 60000
    s = ms // 1000
    ms -= s * 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segments_to_srt(segments: list[dict]) -> str:
    lines = []
    for i, seg in enumerate(segments, 1):
        lines.append(str(i))
        lines.append(f"{_format_srt_ts(seg['start_sec'])} --> {_format_srt_ts(seg['end_sec'])}")
        lines.append(seg['text'])
        lines.append('')
    return '\n'.join(lines).strip() + '\n'

In [ ]:
checkpoint_dir = validate_checkpoint_dir(DRIVE_CHECKPOINT_DIR)
audio_path = Path(AUDIO_PATH)
if not audio_path.exists():
    raise FileNotFoundError(f'Audio file not found: {audio_path}')

print(f'Using checkpoint dir: {checkpoint_dir}')
print('Checkpoint folder entries:', sorted([p.name for p in checkpoint_dir.glob('*')]))

processor = WhisperProcessor.from_pretrained(str(checkpoint_dir))
model = WhisperForConditionalGeneration.from_pretrained(str(checkpoint_dir))
model = model.to(DEVICE)
model.eval()

if hasattr(model, 'generation_config'):
    model.generation_config.task = 'transcribe'
    model.generation_config.language = None

print('Model and processor loaded successfully.')


Using checkpoint dir: /content/drive/MyDrive/University/Y2T2/LLM/AAI3008_artifacts/best_checkpoint_v4
Checkpoint folder entries: ['config.json', 'generation_config.json', 'model.safetensors', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model and processor loaded successfully.


In [ ]:
def decode_chunk(chunk_audio: np.ndarray, sr: int, prompt_lang: str, max_decode_len: int = None):
    if max_decode_len is None:
        max_decode_len = MAX_DECODE_LEN

    feat = processor.feature_extractor(
        chunk_audio,
        sampling_rate=sr,
        return_tensors='pt',
        return_attention_mask=True,
    )
    input_features = feat.input_features.to(model.device)
    attention_mask = feat.attention_mask.to(model.device)

    if prompt_lang == 'en':
        forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(language='english', task='transcribe')
    elif prompt_lang == 'zh':
        forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(language='chinese', task='transcribe')
    else:
        forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(task='transcribe')

    with torch.no_grad():
        pred_ids = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            forced_decoder_ids=forced_decoder_ids,
            max_length=max_decode_len,
            num_beams=5,
            do_sample=False,
        )

    text = processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True).strip()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


audio, sr = load_audio_mono_16k(str(audio_path))
total_sec = len(audio) / sr
spans = chunk_ranges(total_sec, CHUNK_SEC, MIN_CHUNK_SEC)

rows = []
for idx, (s, e) in enumerate(spans):
    st = int(s * sr)
    ed = int(e * sr)
    chunk = audio[st:ed]
    dur = e - s

    mode = resolve_chunk_lang(idx, INFERENCE_MODE)

    if mode in {'en', 'zh'}:
        pred_text = decode_chunk(chunk, sr, mode)
        chosen = f'{mode}_prompt'
    elif mode == 'mixed':
        pred_en = decode_chunk(chunk, sr, 'en')
        pred_zh = decode_chunk(chunk, sr, 'zh')
        pred_text, chosen = choose_mixed(pred_en, pred_zh)
    else:
        # auto: dual decode and pick with mixed-preservation heuristic
        pred_en = decode_chunk(chunk, sr, 'en')
        pred_zh = decode_chunk(chunk, sr, 'zh')
        pred_text, chosen = choose_mixed(pred_en, pred_zh)

    # If output is suspiciously short for long chunk, retry with larger max_length.
    if dur > 12 and len(pred_text) < 20:
        retry_lang = 'zh' if ('zh' in chosen) else 'en'
        pred_retry = decode_chunk(chunk, sr, retry_lang, max_decode_len=448)
        if len(pred_retry) > len(pred_text):
            pred_text = pred_retry
            chosen = chosen + '_retry448'

    rows.append({
        'chunk_idx': idx,
        'start_sec': s,
        'end_sec': e,
        'duration_sec': dur,
        'lang_hint': mode,
        'pred_text': pred_text,
        'chosen_prompt': chosen,
        'detected_bucket': language_bucket(pred_text),
    })

res_df = pd.DataFrame(rows)
raw_transcription = ' '.join(res_df['pred_text'].tolist()).strip()
if MERGE_OVERLAP:
    full_transcription = merge_overlapping_texts(res_df['pred_text'].tolist(), OVERLAP_MAX_TOKENS, OVERLAP_MIN_TOKENS)
else:
    full_transcription = raw_transcription

print('=== Audio Metrics ===')
print(f'File: {audio_path.name}')
print(f'Duration (sec): {total_sec:.2f}')
print(f'Chunks: {len(res_df)} (chunk_sec={CHUNK_SEC}, overlap={CHUNK_OVERLAP_SEC})')
print(f'Inference mode: {INFERENCE_MODE}')
print('Detected language buckets (chunk-level):')
print(dict(Counter(res_df['detected_bucket'].tolist())))

print('\n=== Full Transcription ===')
print(full_transcription)

print('\n=== First 10 chunk details ===')
print(res_df.head(10))

# Optional timestamped segments (per-chunk timestamps for PII bleeping)
segments_df = None
if RETURN_TIMESTAMPS:
    seg_rows = []
    for idx, (s, e) in enumerate(spans):
        st = int(s * sr)
        ed = int(e * sr)
        chunk = audio[st:ed]
        mode = resolve_chunk_lang(idx, INFERENCE_MODE)
        if mode == 'mixed' or mode == 'auto':
            mode = 'zh' if has_zh(res_df.loc[idx, 'pred_text']) else 'en'
        segs = decode_chunk_with_timestamps(processor, model, chunk, sr, mode, s, e)
        for seg in segs:
            seg['chunk_idx'] = idx
            seg['lang_hint'] = mode
            seg_rows.append(seg)
    segments_df = pd.DataFrame(seg_rows)

# Optional save outputs
out_txt = Path('/content/transcription_output.txt')
out_csv = Path('/content/transcription_chunks.csv')
out_txt.write_text(full_transcription, encoding='utf-8')
res_df.to_csv(out_csv, index=False, encoding='utf-8')
print(f'\nSaved full transcription: {out_txt}')
print(f'Saved chunk table: {out_csv}')

if segments_df is not None and SAVE_SEGMENT_FILES:
    seg_csv = Path('/content/transcription_segments.csv')
    seg_srt = Path('/content/transcription_segments.srt')
    segments_df.to_csv(seg_csv, index=False, encoding='utf-8')
    seg_srt.write_text(segments_to_srt(segments_df.to_dict('records')), encoding='utf-8')
    print(f'Saved timestamped segments: {seg_csv}')
    print(f'Saved timestamped SRT: {seg_srt}')

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


=== Audio Metrics ===
File: tsl_audio_test.m4a
Duration (sec): 696.25
Chunks: 39 (chunk_sec=20.0, overlap=2.0)
Inference mode: auto
Detected language buckets (chunk-level):
{'en': 38, 'zh': 1}

=== Full Transcription ===
TheSmartLocal.com Hello! So, welcome to the 9 episode of Before After. We're going to explore places in Singapore today, but I don't know where we're going. I honestly have no idea because the producers did not tell us anything prior to this, and like, when in any way, anything prior to this. And like, anyway, I don't know where, but I gotta go. So, producer, where are we going? Okay, so today, you guys will be exploring some old spots in Singapore. Oh, okay, wait, wait, wait. What do you mean by old spots? Like, spots for elderlies, or like, like, spots for like, like, like... I think old spots, not old people's ones. Oh my god! Eh, Sun Ngai Road. Okay wait, Sun Ngai Road is... I only know Sun Ngai Road, Lux. If I go and search Kampong on Google, this picture will com

## Notes
- This notebook constrains decoding with EN and ZH prompts and selects output with a script-preserving heuristic.
- For purely English audio, outputs should be mostly English; for Chinese audio, mostly Chinese; mixed should retain both where present.
- If you need strict single-language output, replace dual decode with fixed `prompt_lang='en'` or `'zh'` only.


In [ ]:
# Baseline comparison: fine-tuned checkpoint vs baseline Whisper

from pathlib import Path
import pandas as pd


def decode_chunk_with_model(proc, mdl, chunk_audio: np.ndarray, sr: int, prompt_lang: str, max_decode_len: int = None):
    if max_decode_len is None:
        max_decode_len = MAX_DECODE_LEN

    feat = proc.feature_extractor(
        chunk_audio,
        sampling_rate=sr,
        return_tensors='pt',
        return_attention_mask=True,
    )
    input_features = feat.input_features.to(mdl.device)
    attention_mask = feat.attention_mask.to(mdl.device)

    if prompt_lang == 'en':
        forced_decoder_ids = proc.tokenizer.get_decoder_prompt_ids(language='english', task='transcribe')
    elif prompt_lang == 'zh':
        forced_decoder_ids = proc.tokenizer.get_decoder_prompt_ids(language='chinese', task='transcribe')
    else:
        forced_decoder_ids = proc.tokenizer.get_decoder_prompt_ids(task='transcribe')

    with torch.no_grad():
        pred_ids = mdl.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            forced_decoder_ids=forced_decoder_ids,
            max_length=max_decode_len,
            num_beams=5,
            do_sample=False,
        )

    text = proc.tokenizer.decode(pred_ids[0], skip_special_tokens=True).strip()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def transcribe_with_model(proc, mdl, audio_arr: np.ndarray, sr: int, spans, mode: str = 'auto'):
    rows = []
    mode = (mode or 'auto').lower()

    for idx, (s, e) in enumerate(spans):
        st = int(s * sr)
        ed = int(e * sr)
        chunk = audio_arr[st:ed]
        dur = e - s

        local_mode = resolve_chunk_lang(idx, mode)

        if local_mode in {'en', 'zh'}:
            pred_text = decode_chunk_with_model(proc, mdl, chunk, sr, local_mode)
            chosen = f'{local_mode}_prompt'
        elif local_mode == 'mixed':
            pred_en = decode_chunk_with_model(proc, mdl, chunk, sr, 'en')
            pred_zh = decode_chunk_with_model(proc, mdl, chunk, sr, 'zh')
            pred_text, chosen = choose_mixed(pred_en, pred_zh)
        else:
            pred_en = decode_chunk_with_model(proc, mdl, chunk, sr, 'en')
            pred_zh = decode_chunk_with_model(proc, mdl, chunk, sr, 'zh')
            pred_text, chosen = choose_mixed(pred_en, pred_zh)

        if dur > 12 and len(pred_text) < 20:
            retry_lang = 'zh' if ('zh' in chosen) else 'en'
            pred_retry = decode_chunk_with_model(proc, mdl, chunk, sr, retry_lang, max_decode_len=448)
            if len(pred_retry) > len(pred_text):
                pred_text = pred_retry
                chosen = chosen + '_retry448'

        rows.append({
            'chunk_idx': idx,
            'start_sec': s,
            'end_sec': e,
            'duration_sec': dur,
            'pred_text': pred_text,
            'chosen_prompt': chosen,
            'detected_bucket': language_bucket(pred_text),
        })

    df = pd.DataFrame(rows)
    raw_full = ' '.join(df['pred_text'].tolist()).strip()
    if MERGE_OVERLAP:
        full_text = merge_overlapping_texts(df['pred_text'].tolist(), OVERLAP_MAX_TOKENS, OVERLAP_MIN_TOKENS)
    else:
        full_text = raw_full
    return df, full_text


# Build shared chunks once
shared_audio, shared_sr = load_audio_mono_16k(str(audio_path))
shared_total_sec = len(shared_audio) / shared_sr
shared_spans = chunk_ranges(shared_total_sec, CHUNK_SEC, MIN_CHUNK_SEC)

# Fine-tuned inference (uses currently loaded model/processor)
ft_df, ft_full = transcribe_with_model(processor, model, shared_audio, shared_sr, shared_spans, mode=INFERENCE_MODE)

# Baseline inference
baseline_ckpt = BASELINE_MODEL_ID
base_processor = WhisperProcessor.from_pretrained(baseline_ckpt)
base_model = WhisperForConditionalGeneration.from_pretrained(baseline_ckpt).to(DEVICE)
base_model.eval()
if hasattr(base_model, 'generation_config'):
    base_model.generation_config.task = 'transcribe'
    base_model.generation_config.language = None

base_df, base_full = transcribe_with_model(base_processor, base_model, shared_audio, shared_sr, shared_spans, mode=INFERENCE_MODE)

# Summary comparison
print('=== Comparison Summary ===')
print(f'Audio duration: {shared_total_sec:.2f}s | chunks: {len(shared_spans)} | mode: {INFERENCE_MODE}')
print('\nFine-tuned bucket distribution:')
print(dict(Counter(ft_df['detected_bucket'].tolist())))
print('\nBaseline bucket distribution:')
print(dict(Counter(base_df['detected_bucket'].tolist())))

print('\nTranscription length (characters):')
print({'fine_tuned': len(ft_full), 'baseline': len(base_full)})

# Optional WER/CER if reference transcript provided
if REFERENCE_TEXT_PATH and Path(REFERENCE_TEXT_PATH).exists():
    ref_text = Path(REFERENCE_TEXT_PATH).read_text(encoding='utf-8').strip()
    ref_norm = re.sub(r'\s+', ' ', ref_text).strip()
    ft_norm = re.sub(r'\s+', ' ', ft_full).strip()
    base_norm = re.sub(r'\s+', ' ', base_full).strip()

    ft_wer = 100 * jiwer_wer([_to_wer_tokens_local(ref_norm)], [_to_wer_tokens_local(ft_norm)])
    ft_cer = 100 * jiwer_cer([ref_norm], [ft_norm])

    base_wer = 100 * jiwer_wer([_to_wer_tokens_local(ref_norm)], [_to_wer_tokens_local(base_norm)])
    base_cer = 100 * jiwer_cer([ref_norm], [base_norm])

    print('\n=== Reference-based Metrics ===')
    print({'fine_tuned_wer': round(ft_wer, 2), 'fine_tuned_cer': round(ft_cer, 2)})
    print({'baseline_wer': round(base_wer, 2), 'baseline_cer': round(base_cer, 2)})
else:
    print('\nNo REFERENCE_TEXT_PATH provided; skipping WER/CER comparison.')

# Show first few chunk-level side-by-side examples
show_n = min(8, len(ft_df), len(base_df))
print(f'\n=== First {show_n} chunk comparison (trimmed) ===')
for i in range(show_n):
    print(f"\n[Chunk {i}] {ft_df.loc[i, 'start_sec']:.1f}-{ft_df.loc[i, 'end_sec']:.1f}s")
    print('FT  :', ft_df.loc[i, 'pred_text'][:180])
    print('BASE:', base_df.loc[i, 'pred_text'][:180])

# Save outputs
Path('/content/ft_full_transcription.txt').write_text(ft_full, encoding='utf-8')
Path('/content/base_full_transcription.txt').write_text(base_full, encoding='utf-8')
ft_df.to_csv('/content/ft_chunk_predictions.csv', index=False, encoding='utf-8')
base_df.to_csv('/content/base_chunk_predictions.csv', index=False, encoding='utf-8')
print('\nSaved: /content/ft_full_transcription.txt')
print('Saved: /content/base_full_transcription.txt')
print('Saved: /content/ft_chunk_predictions.csv')
print('Saved: /content/base_chunk_predictions.csv')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

=== Comparison Summary ===
Audio duration: 696.25s | chunks: 39 | mode: auto

Fine-tuned bucket distribution:
{'en': 38, 'zh': 1}

Baseline bucket distribution:
{'en': 35, 'zh': 2, 'mixed': 2}

Transcription length (characters):
{'fine_tuned': 16772, 'baseline': 22675}

No REFERENCE_TEXT_PATH provided; skipping WER/CER comparison.

=== First 8 chunk comparison (trimmed) ===

[Chunk 0] 0.0-20.0s
FT  : TheSmartLocal.com
BASE: TheSmartLocal.com

[Chunk 1] 18.0-38.0s
FT  : Hello! So, welcome to the 9 episode of Before After. We're going to explore places in Singapore today, but I don't know where we're going. I honestly have no idea because the produ
BASE: Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! Helo! 

[Chunk 2] 36.0-56.0s
FT  : anything prior to this. And like, anyway, I don't know where, but I gotta go. So, producer, where are we going? Okay, so today, yo

### Optional: WhisperX alignment for tighter timestamps\n
WhisperX can refine segment boundaries and produce tighter word/segment timestamps. This is optional and requires installing extra deps.\n
If your fine-tuned checkpoint is not directly supported, use a baseline Whisper model for alignment only.\n

In [ ]:
# !pip install -q git+https://github.com/m-bain/whisperx.git\n
# import whisperx\n
# device = 'cuda' if torch.cuda.is_available() else 'cpu'\n
# audio_wx = whisperx.load_audio(str(audio_path))\n
# wx_model_id = BASELINE_MODEL_ID  # or a supported Whisper checkpoint path\n
# model_wx = whisperx.load_model(wx_model_id, device=device, compute_type='float16')\n
# result = model_wx.transcribe(audio_wx, batch_size=16)\n
# align_model, metadata = whisperx.load_align_model(language_code='en', device=device)\n
# aligned = whisperx.align(result['segments'], align_model, metadata, audio_wx, device)\n
# aligned_segments = aligned['segments']\n
# print(aligned_segments[:3])\n